# Multi-Encoding Super-Ensemble for PTM Prediction

This notebook creates a **super-ensemble** combining:
- **12 ML models**: RF + XGBoost + SVM trained on 4 encodings each (onehot, blosum, aapc, hybrid)
- **2-3 DL models** (optional): CNN + GRU trained on raw sequences

## Why Multi-Encoding Ensemble?

Different encodings capture different patterns:
- **One-Hot**: Position-specific amino acid identity
- **BLOSUM62**: Evolutionary similarity between amino acids
- **AAPC**: Local dipeptide patterns
- **Hybrid**: BLOSUM + physicochemical properties

Each encoding → Different feature space → Different patterns learned → Better ensemble!

## Expected Performance:
- Single best model: F1 = 0.25-0.35, AUC = 0.78-0.85
- 4-way ensemble (best from each encoding): F1 = 0.35-0.42, AUC = 0.82-0.88
- **10-way super-ensemble (all models)**: F1 = 0.42-0.50, AUC = 0.85-0.92

## 1. Configuration

In [1]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================
import os
# Encodings to use
ENCODINGS = ["onehot", "blosum", "aapc", "hybrid"]

# File paths
VAL_FILE = "../data_engineered/val_with_features_split.csv"  # Original validation file with true labels

# ML model predictions (by encoding)
RF_PRED_TEMPLATE = "../output/rf/{encoding}/rf_{encoding}_val_predictions.csv"
XGB_PRED_TEMPLATE = "../output/xgb/{encoding}/xgb_{encoding}_val_predictions.csv"
KNN_PRED_TEMPLATE = "../output/knn/{encoding}/knn_val_predictions.csv"
#SVM_PRED_TEMPLATE = "../output/svm/{encoding}/svm_val_predictions.csv"

# DL model predictions (optional)
CNN_PREDICTIONS = "../output/cnn/run01_baseline/cnn_val_predictions.csv"
GRU_PREDICTIONS = "../output/gru/run01_baseline/gru_val_predictions.csv"
CRNN_PREDICTIONS = "../output/crnn/run01_baseline/crnn_val_predictions.csv"
TRANSFORMER_GRU_PREDICTIONS = "../output/transformer_gru/run01_frozen_esm2/transformer_gru_val_predictions.csv"

# Output paths
EXPERIMENT_NAME = "run06_rf_xgb_knn_transformer_crnn"
OUTPUT_DIR = f"../output/ensemble/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_WEIGHTS = f"{OUTPUT_DIR}/multi_encoding_ensemble_weights.pkl"
OUTPUT_PREDICTIONS = f"{OUTPUT_DIR}/multi_encoding_ensemble_val_predictions.csv"
OUTPUT_RESULTS = f"{OUTPUT_DIR}/multi_encoding_ensemble_results.csv"

print("Configuration loaded:")
print(f"  Encodings: {ENCODINGS}")
print(f"  Output directory: {OUTPUT_DIR}")

Configuration loaded:
  Encodings: ['onehot', 'blosum', 'aapc', 'hybrid']
  Output directory: ../output/ensemble/run06_rf_xgb_knn_transformer_crnn


## 2. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    roc_auc_score,
    hamming_loss,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 3. Load Ground Truth Labels

In [3]:
# Load validation data with true labels
val_df = pd.read_csv(VAL_FILE)

# Extract labels
label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]
y_true = val_df[label_cols].values

print(f"Validation samples: {len(val_df):,}")
print(f"Labels shape: {y_true.shape}")
print(f"\nClass distribution:")
for idx, label in enumerate(label_cols):
    pos_count = y_true[:, idx].sum()
    print(f"  {label}: {pos_count:,} positive ({pos_count/len(y_true)*100:.2f}%)")

Validation samples: 17,802
Labels shape: (17802, 3)

Class distribution:
  S-glutathionylation: 953 positive (5.35%)
  S-nitrosylation: 2,225 positive (12.50%)
  S-palmitoylation: 647 positive (3.63%)


## 4. Load Model Predictions

Load predictions from all available models:
- RF models (4 encodings)
- XGBoost models (4 encodings)
- CNN (optional)
- GRU (optional)

In [ ]:
predictions = {}

print("="*80)
print("LOADING MODEL PREDICTIONS")
print("="*80)

# Load RF predictions for each encoding
print("\nRandom Forest models:")
for encoding in ENCODINGS:
    pred_file = RF_PRED_TEMPLATE.format(encoding=encoding)
    if os.path.exists(pred_file):
        rf_df = pd.read_csv(pred_file)
        predictions[f"rf_{encoding}"] = rf_df[
            ["rf_glut_proba", "rf_nitro_proba", "rf_palm_proba"]
        ].values
        print(f"  ✓ Loaded RF ({encoding})")
    else:
        print(f"  ✗ Missing: {pred_file}")

# Load XGBoost predictions for each encoding
print("\nXGBoost models:")
for encoding in ENCODINGS:
    pred_file = XGB_PRED_TEMPLATE.format(encoding=encoding)
    if os.path.exists(pred_file):
        xgb_df = pd.read_csv(pred_file)
        predictions[f"xgb_{encoding}"] = xgb_df[
            ["xgb_glut_proba", "xgb_nitro_proba", "xgb_palm_proba"]
        ].values
        print(f"  ✓ Loaded XGBoost ({encoding})")
    else:
        print(f"  ✗ Missing: {pred_file}")

# Load SVM predictions for each encoding
print("\nKNN models:")
for encoding in ENCODINGS:
    pred_file = KNN_PRED_TEMPLATE.format(encoding=encoding)
    if os.path.exists(pred_file):
        knn_df = pd.read_csv(pred_file)
        predictions[f"knn_{encoding}"] = knn_df[
            ["knn_glut_proba", "knn_nitro_proba", "knn_palm_proba"]
        ].values
        print(f"  ✓ Loaded KNN ({encoding})")
    else:
        print(f"  ✗ Missing: {pred_file}")

In [ ]:
# Load CRNN predictions
if os.path.exists(CRNN_PREDICTIONS):
    crnn_df = pd.read_csv(CRNN_PREDICTIONS)
    predictions["crnn"] = crnn_df[
        ["crnn_glut_proba", "crnn_nitro_proba", "crnn_palm_proba"]
    ].values
    print(f"  ✓ Loaded CRNN")
else:
    print(f"  ✗ Missing: {CRNN_PREDICTIONS} (optional)")

In [ ]:
# Load CNN predictions (optional)
print("\nDeep Learning models:")
if os.path.exists(CNN_PREDICTIONS):
    cnn_df = pd.read_csv(CNN_PREDICTIONS)
    predictions["cnn"] = cnn_df[
        ["cnn_glut_proba", "cnn_nitro_proba", "cnn_palm_proba"]
    ].values
    print(f"  ✓ Loaded CNN")
else:
    print(f"  ✗ Missing: {CNN_PREDICTIONS} (optional)")

In [ ]:
# Load GRU predictions (optional)
if os.path.exists(GRU_PREDICTIONS):
    gru_df = pd.read_csv(GRU_PREDICTIONS)
    predictions["gru"] = gru_df[
        ["gru_glut_proba", "gru_nitro_proba", "gru_palm_proba"]
    ].values
    print(f"  ✓ Loaded GRU")
else:
    print(f"  ✗ Missing: {GRU_PREDICTIONS} (optional)")

print(f"\n{'='*80}")
print(f"✓ Total models loaded: {len(predictions)}")
print(f"{'='*80}")

In [ ]:
# Load TRANSFORMER(ESM2)_GRU predictions (optional)
if os.path.exists(TRANSFORMER_GRU_PREDICTIONS):
    transformer_gru_df = pd.read_csv(TRANSFORMER_GRU_PREDICTIONS)
    predictions["transformer_gru"] = transformer_gru_df[
        ["transformer_gru_glut_proba", "transformer_gru_nitro_proba", "transformer_gru_palm_proba"]
    ].values
    print(f"  ✓ Loaded TRANSFORMERR_GRU")
else:
    print(f"  ✗ Missing: {TRANSFORMER_GRU_PREDICTIONS} (optional)")

print(f"\n{'='*80}")
print(f"✓ Total models loaded: {len(predictions)}")
print(f"{'='*80}")

In [ ]:
# Show loaded models
print("\nLoaded models:")
for i, model_name in enumerate(predictions.keys(), 1):
    print(f"  {i}. {model_name}")

## 5. Evaluate Individual Models

Before ensembling, let's see how each individual model performs.

In [ ]:
def evaluate_predictions(y_true, y_pred_proba, threshold=0.5):
    """Evaluate predictions and return metrics."""
    y_pred = (y_pred_proba > threshold).astype(int)
    
    results = {}
    for idx, label in enumerate(label_cols):
        f1 = f1_score(y_true[:, idx], y_pred[:, idx], zero_division=0)
        precision = precision_score(y_true[:, idx], y_pred[:, idx], zero_division=0)
        recall = recall_score(y_true[:, idx], y_pred[:, idx], zero_division=0)
        auc = roc_auc_score(y_true[:, idx], y_pred_proba[:, idx])
        mcc = matthews_corrcoef(y_true[:, idx], y_pred[:, idx])
        results[label] = {
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "auc": auc,
            "mcc": mcc
        }
    
    macro_f1 = np.mean([r["f1"] for r in results.values()])
    macro_precision = np.mean([r["precision"] for r in results.values()])
    macro_recall = np.mean([r["recall"] for r in results.values()])
    macro_auc = np.mean([r["auc"] for r in results.values()])
    macro_mcc = np.mean([r["mcc"] for r in results.values()])
    
    return macro_f1, macro_precision, macro_recall, macro_auc, macro_mcc, results

print("="*80)
print("INDIVIDUAL MODEL PERFORMANCE")
print("="*80)

individual_results = {}

print(f"\n{'Model':<25} {'Macro F1':<12} {'Macro Prec':<12} {'Macro Recall':<12} {'Macro AUC':<12} {'Macro MCC':<12}")
print("-" * 97)

for model_name, pred in predictions.items():
    macro_f1, macro_precision, macro_recall, macro_auc, macro_mcc, results = evaluate_predictions(y_true, pred)
    individual_results[model_name] = {
        "f1": macro_f1,
        "precision": macro_precision,
        "recall": macro_recall,
        "auc": macro_auc,
        "mcc": macro_mcc
    }
    print(f"{model_name:<25} {macro_f1:<12.4f} {macro_precision:<12.4f} {macro_recall:<12.4f} {macro_auc:<12.4f} {macro_mcc:<12.4f}")

# Find best models
best_f1_model = max(individual_results.items(), key=lambda x: x[1]['f1'])
best_auc_model = max(individual_results.items(), key=lambda x: x[1]['auc'])

print(f"\n{'='*80}")
print(f"Best F1:  {best_f1_model[0]} ({best_f1_model[1]['f1']:.4f})")
print(f"Best AUC: {best_auc_model[0]} ({best_auc_model[1]['auc']:.4f})")
print(f"{'='*80}")

## 6. Strategy 1: Simple Average Ensemble

Average all model predictions with equal weights.

In [ ]:
print("\n" + "="*80)
print("STRATEGY 1: SIMPLE AVERAGE (EQUAL WEIGHTS)")
print("="*80)

pred_avg = np.mean(list(predictions.values()), axis=0)
f1_avg, precision_avg, recall_avg, auc_avg, mcc_avg, results_avg = evaluate_predictions(y_true, pred_avg)

print(f"\n{'Label':<30} {'F1':<10} {'Precision':<10} {'Recall':<10} {'AUC':<10} {'MCC':<10}")
print("-" * 80)
for label in label_cols:
    print(f"{label:<30} {results_avg[label]['f1']:<10.4f} {results_avg[label]['precision']:<10.4f} {results_avg[label]['recall']:<10.4f} {results_avg[label]['auc']:<10.4f} {results_avg[label]['mcc']:<10.4f}")

print(f"\n{'Macro Average':<30} {f1_avg:<10.4f} {precision_avg:<10.4f} {recall_avg:<10.4f} {auc_avg:<10.4f} {mcc_avg:<10.4f}")

# Calculate improvement over best single model
improvement_f1 = ((f1_avg - best_f1_model[1]['f1']) / best_f1_model[1]['f1']) * 100
improvement_auc = ((auc_avg - best_auc_model[1]['auc']) / best_auc_model[1]['auc']) * 100

print(f"\nImprovement over best single model:")
print(f"  F1:  {improvement_f1:+.2f}%")
print(f"  AUC: {improvement_auc:+.2f}%")

## 7. Strategy 2: Optimized Weights (Grid Search)

Find optimal weights for each model through grid search.

**Note**: With 8-10 models, exhaustive search is expensive. We use coarse grid search.

In [ ]:
print("\n" + "="*80)
print("STRATEGY 2: OPTIMIZED WEIGHTS (GRID SEARCH)")
print("="*80)

model_names = list(predictions.keys())
n_models = len(model_names)

print(f"\nSearching optimal weights for {n_models} models...")
print("This may take a few minutes...\n")

best_f1 = 0
best_weights = None
best_pred = None

# Coarse grid search (step=0.2 for speed)
# Generate weight combinations that sum to 1.0
from itertools import product

# For computational efficiency, use coarse grid
weight_options = [i/5 for i in range(6)]  # [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

# Limit search space by sampling
# Try random sampling if too many models
if n_models <= 4:
    # Exhaustive search for ≤4 models
    weight_combinations = product(weight_options, repeat=n_models)
    total_combinations = len(weight_options) ** n_models
else:
    # Random sampling for >4 models
    np.random.seed(42)
    n_samples = 5000
    weight_combinations = []
    for _ in range(n_samples):
        weights = np.random.dirichlet(np.ones(n_models))
        # Round to nearest 0.05
        weights = np.round(weights * 20) / 20
        weights = weights / weights.sum()  # Renormalize
        weight_combinations.append(weights)
    total_combinations = n_samples

print(f"Testing {total_combinations:,} weight combinations...\n")

tested = 0
for weights_tuple in weight_combinations:
    if n_models <= 4:
        weights = np.array(weights_tuple)
        # Skip if weights don't sum to ~1.0
        if abs(weights.sum() - 1.0) > 0.01:
            continue
    else:
        weights = weights_tuple
    
    # Calculate weighted predictions
    weighted_pred = sum(w * predictions[m] for w, m in zip(weights, model_names))
    
    # Evaluate (only need F1 for optimization)
    f1, precision, recall, auc, mcc, _ = evaluate_predictions(y_true, weighted_pred)
    
    if f1 > best_f1:
        best_f1 = f1
        best_weights = {m: w for m, w in zip(model_names, weights)}
        best_pred = weighted_pred
    
    tested += 1
    if tested % 1000 == 0:
        print(f"  Tested {tested:,} combinations... (best F1 so far: {best_f1:.4f})")

print(f"\n✓ Search complete! Tested {tested:,} combinations.")
print(f"\nBest weights (F1={best_f1:.4f}):")
for model, weight in sorted(best_weights.items(), key=lambda x: -x[1]):
    if weight > 0.01:  # Only show significant weights
        print(f"  {model:<25} {weight:.3f}")

## 8. Final Ensemble Evaluation

Evaluate the optimized ensemble on validation set.

In [ ]:
print("\n" + "="*80)
print("FINAL ENSEMBLE EVALUATION (OPTIMIZED WEIGHTS)")
print("="*80)

f1_opt, precision_opt, recall_opt, auc_opt, mcc_opt, results_opt = evaluate_predictions(y_true, best_pred)

print(f"\n{'Label':<30} {'F1':<10} {'Precision':<10} {'Recall':<10} {'AUC':<10} {'MCC':<10}")
print("-" * 80)
for label in label_cols:
    print(f"{label:<30} {results_opt[label]['f1']:<10.4f} {results_opt[label]['precision']:<10.4f} {results_opt[label]['recall']:<10.4f} {results_opt[label]['auc']:<10.4f} {results_opt[label]['mcc']:<10.4f}")

print(f"\n{'Macro Average':<30} {f1_opt:<10.4f} {precision_opt:<10.4f} {recall_opt:<10.4f} {auc_opt:<10.4f} {mcc_opt:<10.4f}")

# Calculate improvements
improvement_over_best_f1 = ((f1_opt - best_f1_model[1]['f1']) / best_f1_model[1]['f1']) * 100
improvement_over_best_auc = ((auc_opt - best_auc_model[1]['auc']) / best_auc_model[1]['auc']) * 100
improvement_over_avg_f1 = ((f1_opt - f1_avg) / f1_avg) * 100
improvement_over_avg_auc = ((auc_opt - auc_avg) / auc_avg) * 100

print(f"\n{'='*80}")
print("PERFORMANCE COMPARISON")
print("="*80)
print(f"\n{'Configuration':<30} {'Macro F1':<12} {'Macro Prec':<12} {'Macro Recall':<12} {'Macro AUC':<12} {'Macro MCC':<12}")
print("-" * 98)
print(f"{'Best single model':<30} {best_f1_model[1]['f1']:<12.4f} {best_f1_model[1]['precision']:<12.4f} {best_f1_model[1]['recall']:<12.4f} {best_auc_model[1]['auc']:<12.4f} {best_f1_model[1]['mcc']:<12.4f}")
print(f"{'Simple average ensemble':<30} {f1_avg:<12.4f} {precision_avg:<12.4f} {recall_avg:<12.4f} {auc_avg:<12.4f} {mcc_avg:<12.4f}")
print(f"{'Optimized weight ensemble':<30} {f1_opt:<12.4f} {precision_opt:<12.4f} {recall_opt:<12.4f} {auc_opt:<12.4f} {mcc_opt:<12.4f}")

print(f"\nImprovement over best single model:")
print(f"  F1:  {improvement_over_best_f1:+.2f}%")
print(f"  AUC: {improvement_over_best_auc:+.2f}%")

print(f"\nImprovement over simple average:")
print(f"  F1:  {improvement_over_avg_f1:+.2f}%")
print(f"  AUC: {improvement_over_avg_auc:+.2f}%")

## 9. Visualize Results

In [ ]:
# Plot 1: Individual model performance
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Sort models by F1 score
sorted_models = sorted(individual_results.items(), key=lambda x: x[1]['f1'])
model_names_sorted = [m[0] for m in sorted_models]
f1_scores_sorted = [m[1]['f1'] for m in sorted_models]
auc_scores_sorted = [m[1]['auc'] for m in sorted_models]

# F1 scores
axes[0].barh(range(len(model_names_sorted)), f1_scores_sorted, color='skyblue', edgecolor='black')
axes[0].set_yticks(range(len(model_names_sorted)))
axes[0].set_yticklabels(model_names_sorted, fontsize=8)
axes[0].set_xlabel('Macro F1-score')
axes[0].set_title('Individual Model Performance (F1)')
axes[0].axvline(f1_opt, color='red', linestyle='--', linewidth=2, label=f'Ensemble: {f1_opt:.4f}')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# AUC scores
axes[1].barh(range(len(model_names_sorted)), auc_scores_sorted, color='coral', edgecolor='black')
axes[1].set_yticks(range(len(model_names_sorted)))
axes[1].set_yticklabels(model_names_sorted, fontsize=8)
axes[1].set_xlabel('Macro AUC')
axes[1].set_title('Individual Model Performance (AUC)')
axes[1].axvline(auc_opt, color='red', linestyle='--', linewidth=2, label=f'Ensemble: {auc_opt:.4f}')
axes[1].legend()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/individual_model_performance.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved plot to {OUTPUT_DIR}/individual_model_performance.png")

In [ ]:
# Plot 2: Model weights
fig, ax = plt.subplots(figsize=(12, 6))

# Filter and sort weights
significant_weights = {k: v for k, v in best_weights.items() if v > 0.01}
sorted_weights = sorted(significant_weights.items(), key=lambda x: -x[1])
weight_names = [w[0] for w in sorted_weights]
weight_values = [w[1] for w in sorted_weights]

ax.barh(range(len(weight_names)), weight_values, color='lightgreen', edgecolor='black')
ax.set_yticks(range(len(weight_names)))
ax.set_yticklabels(weight_names)
ax.set_xlabel('Weight')
ax.set_title('Optimized Model Weights in Final Ensemble')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(weight_values):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ensemble_weights.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved plot to {OUTPUT_DIR}/ensemble_weights.png")

## 10. Save Ensemble Outputs

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Save optimal weights
with open(OUTPUT_WEIGHTS, "wb") as f:
    pickle.dump(best_weights, f)
print(f"✓ Saved optimal weights to {OUTPUT_WEIGHTS}")

# Save ensemble predictions
pred_df = pd.DataFrame({
    "ID": val_df["ID"],
    "ensemble_glut_proba": best_pred[:, 0],
    "ensemble_nitro_proba": best_pred[:, 1],
    "ensemble_palm_proba": best_pred[:, 2]
})
pred_df.to_csv(OUTPUT_PREDICTIONS, index=False)
print(f"✓ Saved ensemble predictions to {OUTPUT_PREDICTIONS}")

# Save results summary
results_summary = pd.DataFrame({
    "model": ["best_single", "simple_average", "optimized_ensemble"],
    "macro_f1": [best_f1_model[1]['f1'], f1_avg, f1_opt],
    "macro_precision": [best_f1_model[1]['precision'], precision_avg, precision_opt],
    "macro_recall": [best_f1_model[1]['recall'], recall_avg, recall_opt],
    "macro_auc": [best_f1_model[1]['auc'], auc_avg, auc_opt],
    "macro_mcc": [best_f1_model[1]['mcc'], mcc_avg, mcc_opt]
})
results_summary.to_csv(OUTPUT_RESULTS, index=False)
print(f"✓ Saved results summary to {OUTPUT_RESULTS}")

print("\n" + "="*80)
print("✓ MULTI-ENCODING SUPER-ENSEMBLE COMPLETE!")
print("="*80)
print(f"\nFinal Performance:")
print(f"  Macro F1:        {f1_opt:.4f}")
print(f"  Macro Precision: {precision_opt:.4f}")
print(f"  Macro Recall:    {recall_opt:.4f}")
print(f"  Macro AUC:       {auc_opt:.4f}")
print(f"  Macro MCC:       {mcc_opt:.4f}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}/")
print("\nNext steps:")
print("1. Use these ensemble predictions for final submission")
print("2. Apply same weights to test set predictions")
print("3. Consider threshold tuning for further improvement")
print("="*80)